# Credit Risk Analysis Using Machine Learning  
### Logistic Regression • Decision Tree • PCA • K-Means Clustering
## 📕  Introduction
This project analyzes customer credit risk using multiple machine learning techniques.
The goal is to predict whether a customer will default on their loan and to explore hidden
patterns in the dataset using unsupervised learnin
- Total samples:307,511
- Original features:122r.
g.


**Institution:** Taibah University
## Group 4 
- **Layan Mualla Alharqani** — 4450712  
- **Joud Faraj Almutairi** — 4450660  
- **Arwa Saud Alhaidhari** — 4450240  
- **Yara Talal Alharbi** — 4454042  
- **Raneem Abdullah Almukhlafi** — 4452518  


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, precision_score, recall_score,ConfusionMatrixDisplay,
                             f1_score, confusion_matrix, roc_curve, roc_auc_score,classification_report,silhouette_score )

In [ ]:

# 1. Load the data
df = pd.read_csv("application_train.csv")
print("original shape:",df.shape)

# 2. Drop columns with a high percentage of missing values
missing_percent = df.isnull().sum() / len(df) * 100
columns_to_drop = missing_percent[missing_percent > 40].index
df = df.drop(columns=columns_to_drop)
print("dropped columns:", len(columns_to_drop))
print("shape after dropping:",df.shape)

# 3. Fill remaining missing values
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True) #mode for categorical columns
        else:
            df[col].fillna(df[col].median(), inplace=True)   # median for numeric columns
print("missing values handled")

# 4. Convert text (categorical) data to numbers
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    if df[col].nunique() == 2:
        # If the column has only two values (Yes/No)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
    else:
        # If the column has more than two values
        df = pd.get_dummies(df, columns=[col], drop_first=True)

print("after encoding :", df.shape)

# 5. outlier handling (IQR)

numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    if IQR > 0:
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower, upper)

print("Outliers handeled")


# 6. Add age feature

if "DAYS_BIRTH" in df.columns:
    df["AGE_YEARS"] = df["DAYS_BIRTH"] / -365

print("Final shape:", df.shape)

# 7. save cleaned data
df.to_csv("cleanedd_application_train.csv", index=False)
print("saved cleaned file")

In [ ]:


# Split features and target
X = df.drop(columns=['TARGET'])
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression
model = LogisticRegression(random_state=42,class_weight='balanced',  max_iter=1000   )

model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
# print Metrics
print("Metrics:")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")
print(f"AUC: {roc_auc:.3f}")

# classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Will Pay (0)', 'Will Default (1)']))

#  confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=['Will Pay (0)', 'Will Default (1)'])
disp.plot(values_format='d', cmap='Blues')
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

# Calculate ROC and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
#auc=auc(fpr,tpr)

# Plot ROC curve
plt.figure(figsize=(4, 4))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.title('ROC Curve - Logistic Regression ', fontsize=16)
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:


# Decision Tree 
clf = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=3,
    class_weight='balanced',
    random_state=42
)
clf.fit(X_train, y_train)
print('Decision Tree trained successfully')

tree_rules = export_text(clf, feature_names=list(X.columns))
print(tree_rules)

#Plot the tree
plt.figure(figsize=(30, 15))
plot_tree(
    clf,
    feature_names=X.columns,
    class_names=['No', 'Yes'],
    filled=True,
    rounded=True,
    max_depth=8
)
plt.tight_layout()
plt.show()

# Predictions
y_pred_tree = clf.predict(X_test)
y_proba_tree = clf.predict_proba(X_test)[:, 1]

# Metrics
accuracy_tree  = accuracy_score(y_test, y_pred_tree)
recall_tree    = recall_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
f1_tree        = f1_score(y_test, y_pred_tree)
roc_auc_tree   = roc_auc_score(y_test, y_proba_tree)

print("Metrics - Decision Tree:")
print(f"Accuracy: {accuracy_tree:.3f}")
print(f"Precision: {precision_tree:.3f}")
print(f"Recall: {recall_tree:.3f}")
print(f"F1-Score: {f1_tree:.3f}")
print(f"AUC: {roc_auc_tree:.3f}")

# Classification report
print("\nClassification Report - Decision Tree:")
print(classification_report(
    y_test, y_pred_tree,
    target_names=['Will Pay (0)', 'Will Default (1)']
))


In [ ]:
# Confusion Matrix
cm_tree = confusion_matrix(y_test, y_pred_tree)
disp_tree = ConfusionMatrixDisplay(
    confusion_matrix=cm_tree,
    display_labels=['Will Pay (0)', 'Will Default (1)']
)
disp_tree.plot(values_format='d', cmap='Blues')
plt.title("Confusion Matrix - Decision Tree")
plt.show()

# ROC curve
fpr_tree, tpr_tree, thresholds_tree = roc_curve(y_test, y_proba_tree)

plt.figure(figsize=(4,4))
plt.plot(fpr_tree, tpr_tree, lw=2,
         label=f'ROC curve (AUC = {roc_auc_tree:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.title('ROC Curve - Decision Tree', fontsize=16)
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()




In [ ]:
comparison_table = pd.DataFrame({
    "Logistic Regression": [accuracy, precision, recall, f1, roc_auc],
    "Decision Tree": [accuracy_tree, precision_tree, recall_tree, f1_tree, roc_auc_tree]
}, index=["Accuracy", "Precision", "Recall", "F1-Score", "AUC"])

display(comparison_table)

plt.figure(figsize=(4, 4))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
plt.plot(fpr_tree, tpr_tree, color='green', lw=2, label=f'Decision Tree (AUC = {roc_auc_tree:.3f})')
plt.plot([0, 1], [0, 1], color='gray',linestyle='--', label='Random Classifier')

plt.title('ROC Curves Comparison', fontsize=15)
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.show()

In [ ]:


df_pca = df.drop(columns=["SK_ID_CURR", "TARGET"], errors='ignore')
# Convert booleans to numeric
df_pca = df_pca.replace({True: 1, False: 0})
# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)
# PCA full fit
pca = PCA()
pca.fit(X_scaled)
# Explained variance plot
plt.figure(figsize=(4,4))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA - Explained Variance")
plt.grid(True)
plt.show()

# Final PCA with 103 components
pca_final = PCA(n_components=103)
X_pca = pca_final.fit_transform(X_scaled)

print("Original shape:", X_scaled.shape)
print("PCA shape:", X_pca.shape)

In [ ]:
# choose k based on elbow result
k = 3
# train K-Means
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
kmeans.fit(X_pca)
# get cluster labels
clusters = kmeans.labels_

# visualize the clusters
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters)
plt.title(f"K-Means Clusters (k = {k}) after PCA")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.grid(True)
plt.show()

print("Number of samples:", len(clusters))
print("Cluster labels (first 15):", clusters[:15])

# Evaluate K-Means clustering

sil_score = silhouette_score(X_pca, clusters)
inertia = kmeans.inertia_

unique_labels, counts = np.unique(clusters, return_counts=True)

print("K-Means Evaluation (k = 3)")
print("Inertia:", inertia)
print("Silhouette score:", sil_score)
print("Cluster sizes:")
for lbl, cnt in zip(unique_labels, counts):
    print(f"  Cluster {lbl}: {cnt} samples")